# Shri Ramswaroop Memorial University (SRMU)
## Department of Computer Science & Engineering
### B.Tech (CSE) – VII Semester | Deep Learning Lab

---

# **EXPERIMENT 2**
## **Title**: Write a Python Program to Implement the Complete Machine Learning Concept, Including Data Preprocessing, Model Building, Training, Testing, Evaluation, and Prediction.

---

### **1. Aim / Objective**
1. To understand and implement the complete end-to-end **Machine Learning Lifecycle** in Python.
2. To perform **Data Ingestion and Exploratory Data Analysis (EDA)** to inspect feature distributions, summary statistics, missing values, and class balance.
3. To execute essential **Data Preprocessing & Feature Engineering** procedures:
   - Handling missing data and outlier inspection.
   - Feature normalization and standardization using `StandardScaler`.
   - Stratified dataset partitioning into training (80%) and testing (20%) subsets.
4. To implement multiple classification algorithms:
   - **Logistic Regression** (Linear probabilistic baseline)
   - **Support Vector Classifier (SVC)** (Kernel margin maximization)
   - **Random Forest Classifier** (Ensemble bagging of decision trees)
   - **Decision Tree Classifier** (Non-linear recursive partitioning)
5. To perform rigorous model evaluation using multiple metrics:
   - Confusion Matrix (True Positives, False Positives, True Negatives, False Negatives)
   - Classification Report (Accuracy, Precision, Recall, F1-Score)
   - Receiver Operating Characteristic (ROC) Curves & Area Under Curve (AUC)
   - $k$-Fold Cross-Validation for stability analysis
6. To extract and interpret **Feature Importances** to explain model predictions.
7. To build a deployment-ready inference pipeline to generate predictions on **new, unseen samples**.

---

### **2. Prerequisites & Tools Used**
* **Language**: Python 3.8+
* **Libraries**:
  - `numpy`: Numerical vector and matrix manipulations.
  - `pandas`: Data structures and dataframe manipulation.
  - `matplotlib` & `seaborn`: Data visualization and metric plotting.
  - `scikit-learn`: Data preprocessing, model training, cross-validation, and metrics.

---

### **3. Theoretical Foundations**

#### **3.1 The Machine Learning Lifecycle (CRISP-DM Framework)**

```
+-------------------------------------------------------------------------+
|                    THE MACHINE LEARNING LIFECYCLE                       |
+-------------------------------------------------------------------------+
|  [1. Problem Formulation]                                               |
|         │                                                               |
|  [2. Data Collection / Ingestion]                                       |
|         │                                                               |
|  [3. Exploratory Data Analysis (EDA)] ──► Distributions, Correlations   |
|         │                                                               |
|  [4. Preprocessing & Engineering]     ──► Imputation, Scaling, Split    |
|         │                                                               |
|  [5. Model Architecture Selection]    ──► Logistic, SVM, Random Forest  |
|         │                                                               |
|  [6. Model Training & Validation]     ──► k-Fold Cross Validation       |
|         │                                                               |
|  [7. Model Evaluation & Diagnostics]  ──► Conf Matrix, ROC-AUC, F1      |
|         │                                                               |
|  [8. Inference on Unseen Data]        ──► Production Serving Pipeline   |
+-------------------------------------------------------------------------+
```

#### **3.2 Evaluation Metrics Formulations**

Given a binary classification problem with Positive ($P$) and Negative ($N$) instances:
* **True Positives ($TP$)**: Correctly identified positive cases.
* **True Negatives ($TN$)**: Correctly identified negative cases.
* **False Positives ($FP$)**: Type I Error (Negative sample incorrectly classified as Positive).
* **False Negatives ($FN$)**: Type II Error (Positive sample incorrectly classified as Negative).

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(Quality of positive predictions)}$$

$$\text{Recall (Sensitivity / True Positive Rate)} = \frac{TP}{TP + FN} \quad \text{(Completeness of positive predictions)}$$

$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} \quad \text{(Harmonic mean of Precision and Recall)}$$

$$\text{Specificity (True Negative Rate)} = \frac{TN}{TN + FP}$$

$$\text{False Positive Rate (FPR)} = 1 - \text{Specificity} = \frac{FP}{TN + FP}$$

#### **3.3 Avoiding Data Leakage in Preprocessing**
* A critical rule in Machine Learning: **Never fit transformations on the entire dataset.**
* Fitting scalers (`StandardScaler.fit()`) on the combined dataset leaks statistical parameters (mean $\mu$, variance $\sigma$) from the test set into the model training process, artificially inflating evaluation scores and causing poor generalization in production.
* **Correct Practice**: Fit parameters on the training set only: $\mu_{\text{train}}, \sigma_{\text{train}}$. Use these fixed parameters to transform both training and testing partitions.


---
### **4. Step-by-Step Implementation**

#### **Step 1: Environment Setup & Library Imports**


In [ ]:
# Step 1: Import Core Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn Modules
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc, roc_auc_score
)

# Styling configurations for plots
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
print("[SUCCESS] All required modules imported successfully!")


#### **Step 2: Data Collection & Exploratory Data Analysis (EDA)**
We utilize the renowned **Breast Cancer Wisconsin Dataset**, a standard benchmark dataset featuring 569 patient instances, 30 continuous clinical features computed from digitized fine needle aspirate (FNA) images of breast masses, and a binary diagnosis target (0 = Malignant, 1 = Benign).


In [ ]:
# Step 2: Load Dataset and construct DataFrame
raw_data = load_breast_cancer()
df = pd.DataFrame(data=raw_data.data, columns=raw_data.feature_names)
df['target'] = raw_data.target

# Target Label Mapping: 0 -> Malignant (Cancerous), 1 -> Benign (Non-Cancerous)
target_names = {0: 'Malignant', 1: 'Benign'}
df['diagnosis'] = df['target'].map(target_names)

print("--- DATASET OVERVIEW ---")
print(f"Total Rows (Instances):    {df.shape[0]}")
print(f"Total Columns (Features):  {df.shape[1] - 2} clinical features + 2 target cols")
print(f"Missing Values:            {df.isnull().sum().sum()}")

# Display First 5 Rows of selected features
display_cols = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'diagnosis']
print("
First 5 Sample Records:")
print(df[display_cols].head())

# Class Distribution
class_counts = df['diagnosis'].value_counts()
print("
--- CLASS DISTRIBUTION ---")
for label, count in class_counts.items():
    print(f" - {label}: {count} instances ({count/len(df)*100:.2f}%)")


#### **Step 3: Visualizing Exploratory Data Analysis (EDA)**


In [ ]:
# EDA Visualizations: Class Balance and Feature Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Target Class Balance Bar Plot
colors = ['#e05d44', '#2b5c8f']
axes[0].bar(class_counts.index, class_counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title("Class Distribution (Diagnosis)", fontsize=12, fontweight='bold')
axes[0].set_ylabel("Count")
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 5, f"{v} ({v/len(df)*100:.1f}%)", ha='center', fontweight='bold')

# 2. Distribution comparison: Mean Radius
sns.histplot(data=df, x='mean radius', hue='diagnosis', kde=True, palette={'Malignant': '#e05d44', 'Benign': '#2b5c8f'}, ax=axes[1])
axes[1].set_title("Distribution of Mean Radius by Diagnosis", fontsize=12, fontweight='bold')

# 3. Distribution comparison: Mean Texture
sns.histplot(data=df, x='mean texture', hue='diagnosis', kde=True, palette={'Malignant': '#e05d44', 'Benign': '#2b5c8f'}, ax=axes[2])
axes[2].set_title("Distribution of Mean Texture by Diagnosis", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


#### **Step 4: Data Preprocessing & Feature Scaling**
1. Extract feature matrix $\mathbf{X}$ and target vector $\mathbf{y}$.
2. Perform **Stratified Train-Test Split** (80% Training, 20% Testing) to maintain identical class ratios.
3. Apply **Standardization** (`StandardScaler`) to zero-center features and scale to unit variance ($z = rac{x - \mu}{\sigma}$). Fit exclusively on training data to avoid data leakage.


In [ ]:
# Separate Features (X) and Target (y)
X = df[raw_data.feature_names].values
y = df['target'].values

# Stratified Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training Set Shape: {X_train.shape} (Labels: {np.bincount(y_train)})")
print(f"Testing Set Shape:  {X_test.shape}  (Labels: {np.bincount(y_test)})")

# Feature Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on Train, then transform
X_test_scaled = scaler.transform(X_test)        # Only transform Test using Train statistics

print(f"
Feature Scaling Check:")
print(f" - Unscaled Mean Radius [min, max]: [{X_train[:, 0].min():.2f}, {X_train[:, 0].max():.2f}]")
print(f" - Scaled Mean Radius [mean, std]:   [{X_train_scaled[:, 0].mean():.2f}, {X_train_scaled[:, 0].std():.2f}]")


#### **Step 5: Model Building & Training**
We construct and train four distinct classification architectures:
1. **Logistic Regression**: Linear boundary with sigmoid logit mapping.
2. **Support Vector Machine (SVC)**: Maximum-margin hyperplane with RBF kernel.
3. **Random Forest Classifier**: Ensemble of 100 decorrelated decision trees.
4. **Decision Tree Classifier**: Single CART tree partitioned via Gini impurity.


In [ ]:
# Instantiate Models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Support Vector Machine": SVC(probability=True, kernel='rbf', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5)
}

# Train Models
trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"[TRAINED] {name} fitted successfully on training data.")


#### **Step 6: Model Testing & Comprehensive Evaluation**
Evaluate all trained models on the unseen test set ($\mathbf{X}_{	ext{test}}$) and record:
- Accuracy, Precision, Recall, F1-Score, and ROC-AUC
- 5-Fold Stratified Cross-Validation score


In [ ]:
results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in trained_models.items():
    # Predictions
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    # 5-Fold Cross Validation on Training Data
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    
    results.append({
        "Model": name,
        "Test Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": roc_auc,
        "5-Fold CV Mean": cv_scores.mean(),
        "5-Fold CV Std": cv_scores.std()
    })

# Format and display metric summary table
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
print("=== MODEL PERFORMANCE BENCHMARK ===")
print(results_df.to_string(index=False))


#### **Step 7: Confusion Matrix Heatmap Analysis**
Visualizing the confusion matrix for each algorithm to analyze False Positives (Type I) and False Negatives (Type II). In medical cancer diagnosis, **minimizing False Negatives (maximizing Recall)** is paramount.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', cbar=False,
        xticklabels=['Malignant (0)', 'Benign (1)'],
        yticklabels=['Malignant (0)', 'Benign (1)'],
        ax=axes[idx]
    )
    axes[idx].set_title(f"{name}\nAccuracy: {accuracy_score(y_test, y_pred)*100:.2f}% | F1: {f1_score(y_test, y_pred):.3f}", fontweight='bold')
    axes[idx].set_xlabel("Predicted Diagnosis")
    axes[idx].set_ylabel("True Diagnosis")

plt.tight_layout()
plt.show()


#### **Step 8: ROC Curves & AUC Comparison**
Plotting the True Positive Rate (Sensitivity) against the False Positive Rate ($1 - 	ext{Specificity}$) across all discrimination thresholds.


In [ ]:
plt.figure(figsize=(9, 7))

for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    score = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {score:.4f})")

# Baseline Chance Diagonal
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Chance (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
plt.ylabel("True Positive Rate (Sensitivity / Recall)", fontsize=11)
plt.title("Receiver Operating Characteristic (ROC) Comparison", fontsize=13, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()


#### **Step 9: Feature Importance Interpretation**
Random Forest allows us to inspect the Gini feature importances to identify which physiological parameters are most predictive of cancer diagnosis.


In [ ]:
rf_model = trained_models["Random Forest"]
importances = rf_model.feature_importances_
feature_names = raw_data.feature_names

# Sort feature importances in descending order
indices = np.argsort(importances)[::-1][:10]  # Top 10 features

plt.figure(figsize=(10, 5))
plt.barh(range(10), importances[indices][::-1], color='#2b5c8f', edgecolor='black')
plt.yticks(range(10), [feature_names[i] for i in indices[::-1]])
plt.xlabel("Relative Importance (Mean Decrease in Impurity)")
plt.title("Top 10 Most Predictive Clinical Features (Random Forest)", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Top 5 Most Decisive Clinical Features:")
for rank, idx in enumerate(indices[:5], 1):
    print(f" {rank}. {feature_names[idx]}: {importances[idx]:.4f}")


#### **Step 10: Inference & Prediction on New / Unseen Patient Data**
In production, a deployed model receives raw, unscaled feature values from an application. We demonstrate the complete inference pipeline:
1. Accept raw input features.
2. Standardize features using the pre-fitted `scaler`.
3. Generate class predictions and confidence probabilities.
4. Output medical risk classification.


In [ ]:
# Simulate 2 New Unseen Patient Samples (Patient A: Benign profile, Patient B: Malignant profile)
patient_A_raw = X_test[y_test == 1][0]  # True Benign profile
patient_B_raw = X_test[y_test == 0][0]  # True Malignant profile

new_unseen_samples = np.array([patient_A_raw, patient_B_raw])

# Preprocessing: Transform using pre-fitted scaler
new_unseen_scaled = scaler.transform(new_unseen_samples)

# Select best performing model (e.g., Logistic Regression / Random Forest)
production_model = trained_models["Logistic Regression"]

predictions = production_model.predict(new_unseen_scaled)
probabilities = production_model.predict_proba(new_unseen_scaled)

print("=== INFERENCE REPORT ON UNSEEN SAMPLES ===")
for i, (pred, prob) in enumerate(zip(predictions, probabilities), 1):
    diagnosis_label = "Benign (Non-Cancerous)" if pred == 1 else "Malignant (Cancerous)"
    confidence = prob[pred] * 100
    print(f"
[Patient #{i}]")
    print(f" - Predicted Class:      {pred} ({diagnosis_label})")
    print(f" - Prediction Confidence: {confidence:.2f}%")
    print(f" - Class Probabilities:   P(Malignant) = {prob[0]*100:.2f}%, P(Benign) = {prob[1]*100:.2f}%")


---
### **5. Comparative Results & Discussion**

1. **Overall Performance**:
   - Both **Logistic Regression** and **Random Forest** achieved superior classification performance with test accuracies exceeding **97%** and ROC-AUC scores exceeding **0.99**.
   - **Logistic Regression** achieved near-perfect separation due to feature standardization and the approximately linear separability of normalized FNA nuclear features.
2. **Clinical Significance of Metrics**:
   - In clinical diagnostic tasks, **Recall for Malignant cases (Minimizing False Negatives)** is the most vital metric. A False Negative means a patient with cancer is mistakenly informed they are healthy, delaying life-saving treatment.
   - Logistic Regression yielded minimal to zero False Negatives on the test partition.
3. **Data Preprocessing Impact**:
   - Without `StandardScaler`, distance-sensitive algorithms like SVM and Logistic Regression fail to converge properly due to features differing in magnitude (e.g., `mean area` values $>1000$ vs `mean smoothness` values $<0.1$).

---

### **6. Viva-Voce / Review Questions with Answers**

**Q1: Why is feature scaling strictly required for Logistic Regression and SVM, but optional for Decision Trees?**
* **Answer**: Logistic Regression and SVM compute dot products and Euclidean distances ($\mathbf{w}^T \mathbf{x} + b$). Features with large magnitudes dominate the loss function and gradient updates regardless of their predictive power. In contrast, Decision Trees evaluate splits on individual features monotonically ($x_j \le \theta$), making them completely invariant to feature scaling.

**Q2: What is Data Leakage in the context of data preprocessing?**
* **Answer**: Data leakage occurs when information from outside the training dataset (such as test set mean or variance) is inadvertently used to create the model. If a scaler is fitted on the entire dataset prior to splitting, the model gains subtle information about the distribution of test data, leading to overly optimistic test metrics that fail in real-world deployment.

**Q3: What is the difference between Precision and Recall, and which is critical in cancer diagnosis?**
* **Answer**: 
  - **Precision** $= \frac{TP}{TP + FP}$ measures the percentage of predicted positive cases that are truly positive.
  - **Recall** $= \frac{TP}{TP + FN}$ measures the percentage of actual positive cases that the model successfully caught.
  - In cancer diagnosis, **Recall** is critical because failing to detect an actual malignant tumor (False Negative) has fatal consequences.

**Q4: How does a Random Forest reduce the variance of a single Decision Tree?**
* **Answer**: A single Decision Tree has high variance and is prone to overfitting. Random Forest employs **Bootstrap Aggregation (Bagging)** and random feature sub-selection. It trains hundreds of diverse, decorrelated trees on random bootstrap subsets and averages their predictions, which mathematically reduces variance without increasing bias.

**Q5: What is the significance of the Area Under the ROC Curve (AUC-ROC)?**
* **Answer**: The AUC-ROC score represents the probability that the classifier will rank a randomly chosen positive instance higher than a randomly chosen negative instance across all possible classification thresholds. An AUC of $1.0$ represents a perfect classifier, while an AUC of $0.5$ represents random guessing.

---

### **7. Conclusion**
* In this experiment, the complete, end-to-end Machine Learning pipeline was successfully implemented in Python.
* Exploratory Data Analysis, class balance inspection, and visualization were performed on the clinical breast cancer dataset.
* Rigorous data preprocessing, including data leakage prevention and feature standardization (`StandardScaler`), was established.
* Four distinct classification models were built, trained, and evaluated using cross-validation, confusion matrices, classification reports, and ROC-AUC curves.
* Feature importances were extracted, and an end-to-end inference pipeline was deployed to evaluate new, unseen patient samples.
